In [ ]:
# Installation de YOLOv8
!pip install ultralytics

import ultralytics
ultralytics.checks()

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 51.7/112.6 GB disk)


In [ ]:
import os
os.makedirs("/content/VisDrone", exist_ok=True)
os.chdir("/content/VisDrone")

!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip
!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip

print("✅ Téléchargement terminé !")

--2026-05-28 05:35:38--  https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/521807533/9137a849-834e-4c54-b2d5-14d12384f10f?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-28T06%3A34%3A21Z&rscd=attachment%3B+filename%3DVisDrone2019-DET-train.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-28T05%3A34%3A21Z&ske=2026-05-28T06%3A34%3A21Z&sks=b&skv=2018-11-09&sig=J7uOigeO8c05ZTxJphsZEqwm5Q94YASi%2Bjs3rpNlwvo%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3OTk1MDEzOCwibmJmIjoxNzc5OTQ2NTM4LCJwYXRoIjoicmVsZWFzZWF

In [ ]:
import os
train_zip = "/content/VisDrone/VisDrone2019-DET-train.zip"
taille = os.path.getsize(train_zip) / (1024*1024)
print(f"Taille train zip : {taille:.1f} MB")


Taille train zip : 1478.1 MB


In [ ]:
import zipfile

print("Dézippage train...")
with zipfile.ZipFile("/content/VisDrone/VisDrone2019-DET-train.zip", 'r') as z:
    z.extractall("/content/VisDrone/")

print("Dézippage val...")
with zipfile.ZipFile("/content/VisDrone/VisDrone2019-DET-val.zip", 'r') as z:
    z.extractall("/content/VisDrone/")

print("✅ Dézippage terminé !")

Dézippage train...
Dézippage val...
✅ Dézippage terminé !


In [ ]:
from pathlib import Path
from PIL import Image

def convert_visdrone_to_yolo(dataset_path):
    classes_valides = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    annotations_path = Path(dataset_path) / "annotations"
    labels_path = Path(dataset_path) / "labels"
    labels_path.mkdir(exist_ok=True)
    fichiers = list(annotations_path.glob("*.txt"))
    print(f"  → {len(fichiers)} fichiers à convertir...")
    for fichier in fichiers:
        lines_yolo = []
        with open(fichier, "r") as f:
            for line in f.readlines():
                parts = line.strip().split(",")
                if len(parts) < 6:
                    continue
                x, y, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
                classe = int(parts[5])
                if classe not in classes_valides:
                    continue
                img_path = Path(dataset_path) / "images" / (fichier.stem + ".jpg")
                img = Image.open(img_path)
                img_w, img_h = img.size
                x_centre = (x + w / 2) / img_w
                y_centre = (y + h / 2) / img_h
                w_norm = w / img_w
                h_norm = h / img_h
                classe_yolo = classes_valides.index(classe)
                lines_yolo.append(f"{classe_yolo} {x_centre:.6f} {y_centre:.6f} {w_norm:.6f} {h_norm:.6f}\n")
        with open(labels_path / fichier.name, "w") as f:
            f.writelines(lines_yolo)
    print(f"  ✅ Done !")

convert_visdrone_to_yolo("/content/VisDrone/VisDrone2019-DET-train")
convert_visdrone_to_yolo("/content/VisDrone/VisDrone2019-DET-val")
print("✅ Conversion terminée !")

  → 6471 fichiers à convertir...
  ✅ Done !
  → 548 fichiers à convertir...
  ✅ Done !
✅ Conversion terminée !


In [ ]:
config = """
path: /content/VisDrone
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
nc: 10
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""
with open("/content/VisDrone/visdrone.yaml", "w") as f:
    f.write(config)
print("✅ Configuration créée !")

✅ Configuration créée !


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive connecté !")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive connecté !


In [ ]:
from ultralytics import YOLO

modele = YOLO("yolov8n.pt")

modele.train(
    data="/content/VisDrone/visdrone.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,
    project="/content/drive/MyDrive/YoloDrone",
    name="yolo_drone"
)

print("✅ Fine-tuning terminé !")

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/VisDrone/visdrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_drone, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=